#  ⭐ `dim_via_administracao`


In [1]:
%run ../../config/bootstrap.py


from pathlib import Path
from utils import get_project_root  

import re
import pandas as pd
from unidecode import unidecode

In [2]:
project_root = get_project_root() 

# 🥉 Bronze 

Raw Data
as is production, low quality

In [ ]:
path = Path(project_root) / "data/01_bronze/anvisa/Medicamentos/Medicamentos.parquet"
bronze = pd.read_parquet(path)
bronze["VIA_ADMINISTRACAO"] = bronze["VIA_ADMINISTRACAO"].fillna("DESCONHECIDO")

bronze = pd.concat(
    [
        (
            bronze[["VIA_ADMINISTRACAO"]]
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
        (
            bronze[["VIA_ADMINISTRACAO_MAE_PAI"]].rename(columns={"VIA_ADMINISTRACAO_MAE_PAI": "VIA_ADMINISTRACAO"})
            .value_counts()
            .rename("FREQUENCIA_REGISTROS")
            .reset_index()
        ),
    ],
    ignore_index=True,
)

In [4]:
bronze.head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS
0,DESCONHECIDO,307235
1,oral,31059
2,infusão intravenosa,26376
3,intramuscular,19866
4,desconhecida,18764


In [5]:
bronze.VIA_ADMINISTRACAO.nunique()

1839

In [6]:
bronze.to_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";", index=False)

# 🥈 Silver

normalized data, medium quality


In [ ]:
from utils.med_normalize_via_adm import normalizar_via_fuzzy

In [8]:
silver = pd.read_csv(Path(project_root) / "eda/Medicamentos/Medicamentos_VIA_ADM.csv", sep=";") ## Manual Normalization
 
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS'], dtype='object')

In [9]:
silver["VIA_ADMINISTRACAO_CHAVE"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(lambda txt: normalizar_via_fuzzy(txt, score_threshold=80))
)

silver["VIA_ADMINISTRACAO_VALOR"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(
        lambda txt: normalizar_via_fuzzy(
            txt,
            score_threshold=80,
            return_numeric=True,
        )
    )
)

silver["VIA_ADMINISTRACAO_DESCRIPTION"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(
        lambda txt: normalizar_via_fuzzy(
            txt,
            score_threshold=80,
            return_description=True,
        )
    )
)
silver["VIA_ADMINISTRACAO_DESCRIPTION_PT"] = (
    silver["VIA_ADMINISTRACAO"]
    .astype(str)
    .apply(
        lambda txt: normalizar_via_fuzzy(
            txt,
            score_threshold=80,
            return_description_pt=True,
        )
    )
)

In [10]:
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_VALOR', 'VIA_ADMINISTRACAO_DESCRIPTION',
       'VIA_ADMINISTRACAO_DESCRIPTION_PT'],
      dtype='object')

In [11]:
silver.head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT
0,DESCONHECIDO,307235,desconhecido,0,unknown,Desconhecida
1,oral,31059,O,5,oral,Oral
2,infusão intravenosa,26376,P,6,parenteral,"Parenteral (Injeção intramuscular, Injeção intravenosa, Injeção subcutânea)"
3,intramuscular,19866,P,6,parenteral,"Parenteral (Injeção intramuscular, Injeção intravenosa, Injeção subcutânea)"
4,desconhecida,18764,desconhecido,0,unknown,Desconhecida


In [12]:
(
    silver.groupby("VIA_ADMINISTRACAO_CHAVE")["FREQUENCIA_REGISTROS"]
    .sum()
    .reset_index(name="FREQUENCIA_REGISTROS_TOTAL")
).sort_values(by='FREQUENCIA_REGISTROS_TOTAL', ascending=False)

,VIA_ADMINISTRACAO_CHAVE,FREQUENCIA_REGISTROS_TOTAL
10,desconhecido,349186
5,P,144797
4,O,53614
1,Inhal,1929
8,TD,1417
2,Instill,672
0,Implant,556
3,N,323
7,SL,286
9,V,161


In [14]:
silver.query("VIA_ADMINISTRACAO_VALOR == 1").head()

,VIA_ADMINISTRACAO,FREQUENCIA_REGISTROS,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT
53,intrauterina,352,Implant,1,implant,Implante
97,Via subdérmica,97,Implant,1,implant,Implante
215,Subdérmico,18,Implant,1,implant,Implante
281,administrada no braço esquerdo,12,Implant,1,implant,Implante
344,administrado no braço esquerdo,8,Implant,1,implant,Implante


In [16]:
silver.columns

Index(['VIA_ADMINISTRACAO', 'FREQUENCIA_REGISTROS', 'VIA_ADMINISTRACAO_CHAVE',
       'VIA_ADMINISTRACAO_VALOR', 'VIA_ADMINISTRACAO_DESCRIPTION',
       'VIA_ADMINISTRACAO_DESCRIPTION_PT'],
      dtype='object')

In [19]:
hist = silver.sort_values(by='VIA_ADMINISTRACAO_VALOR').drop(columns="FREQUENCIA_REGISTROS")
hist.head()

,VIA_ADMINISTRACAO,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT
1845,desconhecida,desconhecido,0,unknown,Desconhecida
1842,Unknown,desconhecido,0,unknown,Desconhecida
9,E2B R2 Code: 065 - Unknown,desconhecido,0,unknown,Desconhecida
1783,19-01-2021,desconhecido,0,unknown,Desconhecida
63,Não informado,desconhecido,0,unknown,Desconhecida


In [20]:
hist.to_parquet(Path(project_root) / "data/02_silver/hist_via_adm/hist_via_adm.parquet", index=False)

# 🥇 Gold


In [22]:
dim = (
    hist[[ "VIA_ADMINISTRACAO_VALOR","VIA_ADMINISTRACAO_CHAVE","VIA_ADMINISTRACAO_DESCRIPTION","VIA_ADMINISTRACAO_DESCRIPTION_PT"]]
    .drop_duplicates()
    .sort_values(by="VIA_ADMINISTRACAO_VALOR")
) 
dim

,VIA_ADMINISTRACAO_VALOR,VIA_ADMINISTRACAO_CHAVE,VIA_ADMINISTRACAO_DESCRIPTION,VIA_ADMINISTRACAO_DESCRIPTION_PT
1845,0,desconhecido,unknown,Desconhecida
1575,1,Implant,implant,Implante
648,2,Inhal,inhalation,Inalatória (pelas vias respiratórias)
1795,3,Instill,instillation,Ocular (nos olhos)
1535,4,N,nasal,Nasal
1844,5,O,oral,Oral
2,6,P,parenteral,"Parenteral (Injeção intramuscular, Injeção intravenosa, Injeção subcutânea)"
1147,7,R,rectal,Retal (pelo ânus)
1827,8,SL,sublingual/buccal/oromucosal,Sublingual/Bucal/Oromucosal
760,9,TD,transdermal,Dérmica (na pele)


In [24]:
dim.to_csv(Path(project_root) / "data/03_gold/dim_via_adm/dim_via_adm.csv", sep=";", index=False)